In [ ]:
# Setup and IBM backend connection

from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from time import sleep
from broadcasting import generate_qiskit_circuit, add_fidelity
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from datetime import datetime

np.set_printoptions(linewidth=200, precision=3, suppress=True)

service = QiskitRuntimeService(name="mprest1")

In [ ]:
# Protocol parameters
M_senders = 1
N_receivers = 2
use_qec = True
shots = 4096
seed = 42

# Theta sweep: nt random samples per sender, uniform on [0, 2pi)
nt = 5
rng = np.random.default_rng(seed)
theta_samples = rng.uniform(0, 2 * np.pi, size=(nt, M_senders))

# Tau delay sweep (backend dt units). Use [0] to skip the delay dimension.
tau_values = np.array([0])

# Build one circuit per theta sample, bind tau for each
circuits = []
run_index = []  # (theta_idx, tau_val) for each circuit

for ti, thetas in enumerate(theta_samples):
    qc = generate_qiskit_circuit(
        M_senders, N_receivers, thetas.tolist(),
        use_receiver_qec_513=use_qec,
    )
    qc, reg_name, phi = add_fidelity(qc, N=N_receivers, thetas=thetas.tolist())
    tau_param = next(p for p in qc.parameters if p.name == "tau")

    for tau in tau_values:
        bound = qc.assign_parameters({tau_param: int(tau)})
        circuits.append(bound)
        run_index.append((ti, int(tau)))

print(f"M={M_senders}, N={N_receivers}, QEC={use_qec}")
print(f"nt={nt} theta samples, {len(tau_values)} tau values -> {len(circuits)} circuits")
print(f"Theta samples (radians):")
for ti, thetas in enumerate(theta_samples):
    print(f"  [{ti}] {thetas}")

In [ ]:
# Transpile and submit to backend
backend = service.least_busy(simulator=False, operational=True)
print(f"Backend: {backend.name}")

pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_circuits = pm.run(circuits)

sampler = Sampler(mode=backend)
pubs = [(isc,) for isc in isa_circuits]
job = sampler.run(pubs, shots=shots)
print(f"Job ID: {job.job_id()}")
print(f"Circuits submitted: {len(pubs)}")
sleep(10)
print(f"Status: {job.status()}")

In [ ]:
# Extract results, compute fidelities, save to file
results = job.result()
all_results = {}

for idx, ((ti, tau), pub_result) in enumerate(zip(run_index, results)):
    fid_data = getattr(pub_result.data, reg_name)
    counts = fid_data.get_counts()
    total = sum(counts.values())

    # BitArray.get_counts() returns per-register bitstrings (no spaces).
    # Qiskit convention: creg[i] = position i from the RIGHT of the bitstring.
    # So bitstr[-1-i] or equivalently bitstr[N-1-i] gives creg[i].
    fidelities = []
    for i in range(N_receivers):
        p0 = sum(v for bs, v in counts.items() if bs[N_receivers - 1 - i] == "0") / total
        fidelities.append(p0)

    key = f"theta_{ti}_tau_{tau}"
    all_results[key] = {
        "theta_idx": ti,
        "thetas": theta_samples[ti].tolist(),
        "tau": tau,
        "fidelities": fidelities,
        "counts": counts,
    }

# Print results table
thetas_str = "  ".join(f"theta_{j}" for j in range(M_senders))
recv_str = "  ".join(f"recv_{i}" for i in range(N_receivers))
print(f"{'idx':>3}  {'tau':>5}  {thetas_str}  |  {recv_str}")
print("-" * (12 + 8 * M_senders + 3 + 8 * N_receivers))
for key, data in all_results.items():
    ti, tau = data["theta_idx"], data["tau"]
    ts = "  ".join(f"{t:6.3f}" for t in data["thetas"])
    fs = "  ".join(f"{f:.4f}" for f in data["fidelities"])
    print(f"{ti:>3}  {tau:>5}  {ts}  |  {fs}")

# Save to timestamped JSON
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
outfile = results_dir / f"run_{timestamp}.json"

run_data = {
    "timestamp": datetime.now().isoformat(),
    "job_id": job.job_id(),
    "backend": backend.name,
    "shots": shots,
    "seed": seed,
    "protocol": {
        "M": M_senders,
        "N": N_receivers,
        "use_qec": use_qec,
        "nt": nt,
        "theta_samples": theta_samples.tolist(),
        "tau_values": [int(t) for t in tau_values],
    },
    "results": {
        key: {
            "theta_idx": data["theta_idx"],
            "thetas": data["thetas"],
            "tau": data["tau"],
            "fidelities": data["fidelities"],
            "counts": data["counts"],
        }
        for key, data in all_results.items()
    },
}

with open(outfile, "w") as f:
    json.dump(run_data, f, indent=2)
print(f"\nResults saved to {outfile}")

In [ ]:
# Plot fidelity vs theta for each receiver (at tau=0)
plot_tau = 0
plot_data = {key: d for key, d in all_results.items() if d["tau"] == plot_tau}

theta_totals = [np.sum(d["thetas"]) for d in plot_data.values()]
fid_by_recv = [
    [d["fidelities"][i] for d in plot_data.values()]
    for i in range(N_receivers)
]

fig, ax = plt.subplots(figsize=(8, 5))
for i in range(N_receivers):
    ax.scatter(theta_totals, fid_by_recv[i], label=f"Receiver {i}", s=40)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Random (0.5)")
ax.set_xlabel(r"$\sum \theta_j$ (radians)")
ax.set_ylabel("Fidelity P(0)")
ax.set_title(f"Fidelity vs total theta — QEC={use_qec}, tau={plot_tau}, backend={backend.name}")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()